# Notebook 03: End-to-End Pipeline PoC (IndicVoice)

This notebook chains **IndicTrans2** (200M distilled) and **Indic Parler-TTS** to turn English input text directly into intelligible Odia, Hindi, and Bengali speech.

In [ ]:
# Install dependencies (run once in Colab/Kaggle)
!pip install -q transformers sentencepiece protobuf accelerate soundfile
!pip install -q git+https://github.com/AI4Bharat/IndicTransToolkit.git
!pip install -q git+https://github.com/huggingface/parler-tts.git

In [ ]:
import os
import time
import torch
import numpy as np
import soundfile as sf
from IPython.display import Audio, display
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from IndicTransToolkit import IndicProcessor
from parler_tts import ParlerTTSForConditionalGeneration

# os.environ["HF_TOKEN"] = "hf_..."

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if "cuda" in DEVICE else torch.float32
print(f"Active device: {DEVICE} ({DTYPE})")

In [ ]:
# 1. Load Translation Model
MT_ID = "ai4bharat/indictrans2-en-indic-dist-200M"
print("Loading IndicTrans2...")
mt_tokenizer = AutoTokenizer.from_pretrained(MT_ID, trust_remote_code=True)
mt_model = AutoModelForSeq2SeqLM.from_pretrained(MT_ID, trust_remote_code=True, torch_dtype=DTYPE).to(DEVICE)
mt_model.eval()
ip = IndicProcessor(inference=True)

# 2. Load TTS Model
TTS_ID = "ai4bharat/indic-parler-tts"
print("Loading Indic Parler-TTS...")
tts_model = ParlerTTSForConditionalGeneration.from_pretrained(TTS_ID, torch_dtype=DTYPE).to(DEVICE)
tts_model.eval()
sampling_rate = tts_model.config.sampling_rate
tts_tokenizer = AutoTokenizer.from_pretrained(TTS_ID)
tts_desc_tokenizer = AutoTokenizer.from_pretrained(tts_model.config.text_encoder._name_or_path)
print("Both models initialized!")

In [ ]:
def translate_and_speak(english_text, target_lang="odia"):
    t0 = time.time()
    lang_map = {
        "odia": ("ory_Orya", "A female speaker speaks in a clear, natural, moderately paced voice with close recording and no background noise."),
        "hindi": ("hin_Deva", "Divya speaks in a clear, moderately paced voice with a close recording and no background noise."),
        "bengali": ("ben_Beng", "Aditi speaks in a clear, moderately paced voice with a close recording and no background noise.")
    }
    tgt_code, desc = lang_map[target_lang.lower()]
    
    # Translate
    pre = ip.preprocess_batch([english_text], src_lang="eng_Latn", tgt_lang=tgt_code)
    inputs = mt_tokenizer(pre, padding="longest", return_tensors="pt").to(DEVICE)
    with torch.inference_mode():
        out = mt_model.generate(**inputs, num_beams=5, max_length=256)
    translated_text = ip.postprocess_batch(mt_tokenizer.batch_decode(out, skip_special_tokens=True), lang=tgt_code)[0]
    
    t_trans = time.time() - t0
    
    # Synthesize
    p_ids = tts_tokenizer(translated_text, return_tensors="pt").input_ids.to(DEVICE)
    d_ids = tts_desc_tokenizer(desc, return_tensors="pt").input_ids.to(DEVICE)
    with torch.inference_mode():
        tts_out = tts_model.generate(input_ids=d_ids, prompt_input_ids=p_ids)
        
    waveform = tts_out.cpu().numpy().squeeze()
    t_total = time.time() - t0
    
    print(f"English:    {english_text}")
    print(f"Translated: {translated_text}")
    print(f"Timing:     Translation {t_trans:.2f}s | Total {t_total:.2f}s")
    
    return translated_text, waveform

# Test execution
english_sample = "Welcome to the temple city of Bhubaneswar. Have a wonderful stay!"
tr_text, audio = translate_and_speak(english_sample, target_lang="odia")
display(Audio(audio, rate=sampling_rate))